# CIFAR-10 extraction

Train or reload a CIFAR-10 diffusion model, inspect its samples, and audit it with Carlini and SIDE.

Install `requirements.txt` in the notebook kernel environment. `train_config.yaml` configures training; `audit.yaml` controls the attacks.


In [ ]:
import json
import os
import sys
import time
from dataclasses import asdict
from pathlib import Path

# Required before CUDA initializes; also makes repeated seeded samples comparable.
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import matplotlib.pyplot as plt
import torch
import torchvision
import yaml

candidates = [Path.cwd(), *Path.cwd().parents]
example_dir = next(
    (path / 'examples' / 'extraction' / 'cifar10' for path in candidates if (path / 'examples' / 'extraction' / 'cifar10' / 'audit.yaml').exists()),

    Path.cwd(),
)
if not (example_dir / 'train_config.yaml').exists():
    raise FileNotFoundError('Run this notebook from the LeakPro checkout or examples/extraction/cifar10 directory.')
repo_root = example_dir.parents[2]
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(example_dir))
os.chdir(example_dir)

from leakpro import LeakPro
from leakpro.utils.save_load import hash_config
from cifar10_handler import CIFAR10ExtractionHandler, load_audit_config
from cifar10_model import (
    TrainConfig,
    load_cifar10,
    make_adapter,
    make_feature_extractor,
    seed_everything,
    select_device,
    sha256_file,
    sha256_module_state,
    sha256_tensor,
    train_or_load_target,
)

train_config = yaml.safe_load(Path('train_config.yaml').read_text(encoding='utf-8'))
train = TrainConfig(seed=int(train_config['run']['random_seed']), **train_config['train'])
audit_config_value = os.getenv(
    'LEAKPRO_CIFAR_AUDIT_CONFIG',
    train_config['run']['audit_config'],
)
audit_config_path = Path(audit_config_value)
device = select_device(os.getenv('LEAKPRO_CIFAR_DEVICE', train_config['run']['device']))
seed_everything(train.seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
data_dir = Path(os.getenv('LEAKPRO_CIFAR_DATA_DIR', train_config['run']['data_dir']))
target_dir = Path(os.getenv('LEAKPRO_CIFAR_TARGET_DIR', train_config['run']['target_dir']))
target_dir.mkdir(parents=True, exist_ok=True)
notebook_source_hash = hash_config({'cells': [
    {'cell_type': cell['cell_type'], 'source': cell['source']}
    for cell in json.loads(Path('main.ipynb').read_text(encoding='utf-8'))['cells']
]})
print({
    'device': str(device),
    'data_dir': str(data_dir),
    'target_dir': str(target_dir),
    'audit_config': str(audit_config_path),
})

## Prepare the reference images

The target trains on a deterministic prefix of the CIFAR-10 training set. Both attacks use that prefix as their reference data. SIDE derives its classifier labels from generated images.


In [ ]:
train_dataset, reference_images = load_cifar10(train, target_dir, data_dir)
assert reference_images.shape == (train.reference_size, 3, 32, 32)
assert reference_images.dtype == torch.float32
assert torch.isfinite(reference_images).all()
assert float(reference_images.min()) >= -1.0 and float(reference_images.max()) <= 1.0

figure, axes = plt.subplots(1, 8, figsize=(12, 2))
for axis, image in zip(axes, reference_images[:8].add(1.0).div(2.0)):
    axis.imshow(image.permute(1, 2, 0))
    axis.axis('off')
figure.suptitle(f'Reference prefix: {train.reference_size} CIFAR-10 training images')
plt.show()

feature_extractor, feature_transform = make_feature_extractor()

## Audit and reporting

The handler supplies the trained model and reference images. The notebook records the target identity and saves results to the output directory configured in `audit.yaml`.

Inspect unguided samples and candidate pairs before interpreting their distances.


In [ ]:
def show_nearest_matches(result, title, maximum=6):
    records = [record for record in result.candidates if record.nearest_reference_index is not None]
    records.sort(key=lambda record: record.nearest_reference_distance)
    records = records[:maximum]
    if not records:
        print(f'{title}: no qualifying candidates')
        return
    figure, axes = plt.subplots(len(records), 2, figsize=(4, 2 * len(records)), squeeze=False)
    for row, record in enumerate(records):
        generated = result.images[record.image_index].detach().cpu()
        reference = reference_images[record.nearest_reference_index].add(1.0).div(2.0)
        axes[row, 0].imshow(generated.permute(1, 2, 0).clamp(0.0, 1.0))
        axes[row, 0].set_title(f'generated, d={record.nearest_reference_distance:.4f}')
        axes[row, 1].imshow(reference.permute(1, 2, 0).clamp(0.0, 1.0))
        axes[row, 1].set_title(f'reference #{record.nearest_reference_index}')
        axes[row, 0].axis('off')
        axes[row, 1].axis('off')
    figure.suptitle(title)
    figure.tight_layout()
    plt.show()


def audit_target(model, diffusion, checkpoint_path, training_seconds=0.0):
    checkpoint_hash = sha256_file(checkpoint_path)
    adapter = make_adapter(model, diffusion, device)
    samples = adapter.sample(4, conditions=None, seed=train.seed + 10)
    repeated = adapter.sample(4, conditions=None, seed=train.seed + 10)
    assert samples.shape == (4, 3, 32, 32)
    assert torch.isfinite(samples).all()
    torch.testing.assert_close(samples, repeated)
    figure, axes = plt.subplots(1, 4, figsize=(7, 2))
    for axis, image in zip(axes, samples.detach().cpu().add(1.0).div(2.0)):
        axis.imshow(image.permute(1, 2, 0).clamp(0.0, 1.0))
        axis.axis('off')
    figure.suptitle('Unguided DDIM samples')
    plt.show()

    identity_components = {
        'target_checkpoint_sha256': checkpoint_hash,
        'model_source_sha256': sha256_file(Path('cifar10_model.py')),
        'handler_source_sha256': sha256_file(Path('cifar10_handler.py')),
        'notebook_source_sha256': notebook_source_hash,
        'sampling_steps': train.sampling_steps,
        'side_feature_state_sha256': sha256_module_state(feature_extractor),
        'side_feature_transform': 'resize-224-bilinear-align-corners-false-imagenet-normalization-v1',
        'authorized_references_sha256': sha256_tensor(reference_images),
    }
    target_hash = f"sha256:{hash_config(identity_components)}"
    CIFAR10ExtractionHandler.configure(adapter=adapter, references=reference_images,
                           feature_extractor=feature_extractor, feature_transform=feature_transform)
    audit_config = load_audit_config(
        audit_config_path,
        target_hash=target_hash,
    )
    audit_output = Path(audit_config['audit']['output_dir'])
    runtime_config_path = target_dir / 'audit.yaml'
    runtime_config_path.write_text(yaml.safe_dump(audit_config, sort_keys=False), encoding='utf-8')
    attack_configs = {entry['attack']: entry for entry in audit_config['audit']['attack_list']}
    audit_started = time.perf_counter()
    results = LeakPro(CIFAR10ExtractionHandler, str(runtime_config_path)).run_audit()
    audit_seconds = time.perf_counter() - audit_started
    assert len(results) == len(attack_configs)
    for result in results:
        if result.metrics.get('mode') == 'unconditional_reference_audit':
            assert result.metrics['images_generated'] == attack_configs['carlini_diffusion']['num_unconditional_generations']
        else:
            assert result.metrics['images_generated'] == attack_configs['side']['num_generations']
            assert result.execution_trace[-1]['guidance_calls'] > 0
        result_dir = audit_output / 'results' / result.id
        assert (result_dir / 'result.json').exists()
        assert (result_dir / 'candidates.npz').exists()
        print(result.name, result.metrics)
        show_nearest_matches(result, result.name)
    manifest = {
        'model_configuration': {key: getattr(train, key) for key in (
            'model_channels', 'num_res_blocks', 'dropout', 'timesteps', 'sampling_steps'
        )},
        'training_configuration': asdict(train),
        'reference_scope': {'dataset': 'CIFAR-10 train', 'selection': 'prefix', 'count': train.reference_size},
        'checkpoint': str(checkpoint_path),
        'checkpoint_sha256': checkpoint_hash,
        'target_hash': target_hash,
        'identity_components': identity_components,
        'device': str(device),
        'torch': torch.__version__,
        'torchvision': torchvision.__version__,
        'training_seconds': training_seconds,
        'audit_seconds': audit_seconds,
        'source_audit_config': str(audit_config_path),
        'runtime_audit_config': str(runtime_config_path),
        'audit_config': audit_config,
        'result_ids': [result.id for result in results],
    }
    manifest_path = target_dir / 'run_manifest.json'
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8')
    print({'manifest': str(manifest_path), 'audit_seconds': round(audit_seconds, 2)})
    return results

## Train and audit

Train a new target or load a compatible checkpoint from `target_dir`. A resume checkpoint is saved every 100 epochs. `force_retrain: true` discards the target and resume checkpoints.

Inspect unguided samples before interpreting extraction results.


In [ ]:
force_retrain_env = os.getenv('LEAKPRO_CIFAR_FORCE_RETRAIN')
force_retrain = force_retrain_env == '1' if force_retrain_env is not None else bool(train_config['run']['force_retrain'])
training_started = time.perf_counter()
model, diffusion, checkpoint_path, epoch_losses = train_or_load_target(
    train, train_dataset, target_dir, device, force_retrain=force_retrain
)
training_seconds = time.perf_counter() - training_started
if epoch_losses:
    plt.figure(figsize=(6, 3))
    plt.plot(epoch_losses)
    plt.xlabel('Epoch')
    plt.ylabel('Improved DDPM hybrid loss')
    plt.yscale('log')
    plt.title('Target training loss')
    plt.show()
results = audit_target(model, diffusion, checkpoint_path, training_seconds=training_seconds)
